In [17]:
import pandas as pd
import yfinance as yf
from ollama import Client

def fetch_nifty_data():
    # Use max allowed by yfinance: '60d' for 5m data
    df = yf.download("^NSEI", interval="5m", period="60d", progress=False, auto_adjust=False)
    if df.empty:
        print("No data fetched for NIFTY 50! Check market hours, network, or symbol.")
        return None
    df = df.dropna()
    return df

def format_candles_for_llm(df, n_candles=50):
    # Clean up columns
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join([str(c) for c in col if c]).lower() for col in df.columns]
    else:
        df.columns = [str(c).lower() for c in df.columns]
    # Get last N candles
    recent = df.tail(n_candles).copy()
    # Keep only standard columns
    keep_cols = ['open', 'high', 'low', 'close', 'volume']
    recent = recent[[col for col in keep_cols if col in recent.columns]]
    # Convert to CSV-like string for LLM
    data_txt = recent.to_csv(index=True)  # index=True to keep datetime
    return f"Recent NIFTY 50 5-minute candles:\n{data_txt}"

def ask_gemma3n(data_txt, model='gemma3n:e2b'):
    prompt = (
        f"{data_txt}\n"
        "Based only on this raw price and technical trends of NIFTY Index, Volume data does not the right data"
        "what should an experienced intraday trader do RIGHT NOW based on data? "
        "Give a short reasoning and a one-word suggestion (Strong Buy, Strong Sell, Neutral)."
        "Target: <number>\n"
        "Stop-Loss: <number>\n"
        "No extra explanation. No filler text."
    )
    client = Client(host='http://localhost:11434')
    try:
        response = client.generate(
            model=model,
            prompt=prompt,
            options={"temperature": 0.5}
        )
        print("\nLLM reply:\n", response['response'])
    except Exception as e:
        print("Ollama/LLM error:", e)

if __name__ == "__main__":
    model_name = "gemma3n:e2b"  # Use your exact Gemma 3n model name!
    n_candles = 50  # Number of recent 5m candles to send; increase/decrease as needed (<= 200 is safe)
    df = fetch_nifty_data()
    if df is not None and not df.empty:
        data_txt = format_candles_for_llm(df, n_candles=n_candles)
        print("Prompt sent to LLM (showing only first 5 rows):\n", "\n".join(data_txt.splitlines()[:7]), "...\n")
        ask_gemma3n(df, model=model_name)
    else:
        print("No data to analyze.")


Prompt sent to LLM (showing only first 5 rows):
 Recent NIFTY 50 5-minute candles:
Datetime
2025-07-04 06:50:00+00:00
2025-07-04 06:55:00+00:00
2025-07-04 07:00:00+00:00
2025-07-04 07:05:00+00:00
2025-07-04 07:10:00+00:00 ...


LLM reply:
 Neutral.

Reasoning: Price action shows a mixed trend with some upward movement but also periods of consolidation and low volume.  The lack of volume suggests limited conviction in either direction.  A trader would need more volume confirmation to make a decisive trade.

Target: 25450
Stop-Loss: 25400



In [12]:
fetch_nifty_data()

Price,Adj Close,Close,High,Low,Open,Volume
Ticker,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI,^NSEI
Datetime,,,,,,
2025-04-09 03:45:00+00:00,22410.550781,22410.550781,22456.150391,22362.500000,22453.250000,0
2025-04-09 03:50:00+00:00,22428.699219,22428.699219,22458.900391,22399.349609,22412.349609,0
2025-04-09 03:55:00+00:00,22385.099609,22385.099609,22440.550781,22368.699219,22429.400391,0
2025-04-09 04:00:00+00:00,22410.699219,22410.699219,22412.849609,22362.099609,22382.900391,0
2025-04-09 04:05:00+00:00,22391.349609,22391.349609,22420.150391,22371.050781,22411.400391,0
...,...,...,...,...,...,...
2025-07-07 03:50:00+00:00,25449.949219,25449.949219,25478.099609,25443.199219,25445.199219,0
2025-07-07 03:55:00+00:00,25439.900391,25439.900391,25452.750000,25433.699219,25450.699219,0
